## Import

In [3]:
import pandas as pd
import psycopg as pg
import os
import json as js
from dotenv import load_dotenv

## Connexion à la DB

In [ ]:
load_dotenv()

connection = pg.connect(
    host="db",
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    port=os.getenv("PORT_DB"),
)

## Conversion en DataFrame

In [5]:
df_entreprise = pd.read_sql("SELECT * FROM entreprise", con=connection)
df_poste = pd.read_sql("SELECT * FROM poste", con=connection)

print("Table entreprise :")
print(df_entreprise.head(), "\n")

print("Table poste :")
print(df_poste.head())

Table entreprise :
   id_entreprise type_contrat        nom_compagnie seo_alias  \
0            200  [permanent]                  Zar      None   
1            242  [permanent]  Tactical Adventures      None   
2            199  [permanent]               Jetdev      None   
3            233  [permanent]                HoppR      None   
4            197  [permanent]      UBIK Ingénierie      None   

                                                logo      type_entreprise  \
0                                            zar.com      product_company   
1                             tacticaladventures.com      product_company   
2                                          jetdev.fr      service_company   
3                                          hoppr.com  consultiung_company   
4  https://ui-avatars.com/api/?name=ubikingenieri...      service_company   

                                  secteur_entreprise  \
0                                               None   
1                    

/tmp/ipykernel_50370/1542705419.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_entreprise = pd.read_sql("SELECT * FROM entreprise", con=connection)
/tmp/ipykernel_50370/1542705419.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_poste = pd.read_sql("SELECT * FROM poste", con=connection)


## Traitement des données

In [ ]:
# je calcule le nombre de candidatures totales
sum_candidature = int(df_poste["nb_postulations"].sum())
print("Nombre de candidatures total :", sum_candidature, "\n")

# je groupe par entreprise 
grouped_poste = df_poste.groupby("id_entreprise", as_index=False)["nb_postulations"].sum().astype(int)

# je calcule la part des candidatures détenues
grouped_poste["part_candidatures"] = (grouped_poste["nb_postulations"] * 100) / sum_candidature

# je trie par ordre décroissant pour avoir les plus importantes
grouped_poste = grouped_poste.sort_values(["part_candidatures"], ascending=False)

# j'ajoute le nom des entreprises
grouped_poste = grouped_poste.merge(df_entreprise[["id_entreprise", "nom_compagnie"]], on="id_entreprise", how="left")

# je sélectionne le minimum d'entreprises qui ensemble détiennent plus de 50% des candidatures
# si elles sont plus de 5, je sélectionne seulement le top 5
pourcentage = 0
dictionnaire = {}
for index, ligne in grouped_poste.iterrows():
    if pourcentage > 50 :
        break
    pourcentage += ligne["part_candidatures"]
    dictionnaire[ligne["nom_compagnie"]] = round(ligne["part_candidatures"], 2)

if len(dictionnaire) > 5 :
    dictionnaire = dict(list(dictionnaire.items())[:5])

# je calcule le pourcentage détenu par les autres entreprises
total = 0
for clé, valeur in dictionnaire.items():
    total += valeur

autre = 100 - total
dictionnaire["Autres"] = autre

# résultat
print("Résultat :", dictionnaire)

Nombre de candidatures total : 2241 

Résultat : {'Jetdev': 27.18, 'HoppR': 16.29, 'UBIK Ingénierie': 11.74, 'Autres': 44.79}


/tmp/ipykernel_50370/2336588118.py:6: FutureWarning: The provided callable <built-in function sum> is currently using np.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string np.sum instead.
  grouped_poste = df_poste.groupby("id_entreprise", as_index=False)["nb_postulations"].apply(sum).astype(int)


## Conversion en liste exploitable par le front

In [7]:
json = []

total = {
    "total": sum_candidature
}
json.append(total)

couleurs_2 = ["#47AD95", "#414141"]
couleurs_3 = ["#47AD95", "#8208D4", "#414141"]
couleurs_4 = ["#47AD95", "#2A9EBD", "#8208D4", "#414141"]
couleurs_5 = ["#47AD95", "#2A9EBD", "#214CC4", "#8208D4", "#414141"]
couleurs_6 = ["#47AD95", "#2A9EBD", "#214CC4", "#8208D4", "#5B0C83", "#414141"]

palette_map = {
    2: couleurs_2,
    3: couleurs_3,
    4: couleurs_4,
    5: couleurs_5,
    6: couleurs_6
}

for i, (clé, valeur) in enumerate(dictionnaire.items()):
    element = {
        "label": clé,
        "value": valeur,
        "color": palette_map[len(dictionnaire)][i]
    }
    json.append(element)

if not os.path.exists("../../Frontend/techyourjob-frontend/public/data/cache"):
    os.makedirs("../../Frontend/techyourjob-frontend/public/data/cache")


frontend_cache_path = "../../Frontend/techyourjob-frontend/public/data/cache/candidatures_entreprise.json"
with open(frontend_cache_path, "w", encoding="utf-8") as f:
    js.dump(json, f, ensure_ascii=False, indent=2)

## Vérification de l'exécution du fichier

In [ ]:
print("Entreprise_Candidature exécuté")